# Quantum phase estimation: from textbook to NISQ reality

*Part of the QUEST Foundations & Algorithms series*

Quantum Phase Estimation (QPE) is the workhorse behind Shor's algorithm, and quantum chemistry leans on it for ground state energies. Any time you need eigenvalue information out of a unitary operator, QPE is the standard tool. Textbooks teach QPE as if you can just add more precision qubits to get more accurate answers. This notebook shows what happens when you try that on real hardware.

We'll set up QPE for an operator whose eigenvalues we know exactly (a phase gate), sweep the number of precision qubits from 2 to 5, and watch two things happen:

1. On the ideal simulator, accuracy improves monotonically as we add precision qubits, just as the textbook promises.
2. On real hardware, accuracy improves up to a point and then *degrades*, because each additional precision qubit adds circuit depth and noise accumulates faster than precision does.

Then we'll introduce Zero-Noise Extrapolation (ZNE), a widely-used error mitigation technique, and show how it partially recovers the ideal behavior at the cost of extra shots.

**Learning objectives.** By the end of this notebook you will:

1. Implement QPE for a known test operator.
2. Understand the theoretical relationship between precision qubits and accuracy.
3. Observe how NISQ noise limits the practical depth of QPE circuits.
4. Apply zero-noise extrapolation as a first-class error mitigation technique.
5. Reason about when QPE is the right tool and when variational alternatives (like VQE) are better.

**What to bring in.** You should have seen QPE at least once in class. If not, the review section below sketches enough to follow along, but the deeper insights land better after your lecture.

**Credit budget.** This notebook consumes approximately 2,500 shots on hardware for the precision sweep, plus another 3,000 shots for the ZNE demonstration. Total on the order of $5,500$ shots, which is a few dollars in credits.


## QPE in two sentences

Given a unitary $U$ and one of its eigenstates $|\psi\rangle$ satisfying $U|\psi\rangle = e^{2\pi i \phi}|\psi\rangle$, QPE estimates the phase $\phi \in [0, 1)$ using $t$ **precision qubits** (also called counting qubits). The estimator returns a binary approximation of $\phi$ with $t$ bits of precision.

The circuit has four steps:

1. Put the precision register into uniform superposition (Hadamards).
2. Apply controlled-$U^{2^j}$ from precision qubit $j$ to the target register, for each $j$.
3. Apply the inverse Quantum Fourier Transform to the precision register.
4. Measure.

Each additional precision qubit doubles the accuracy (halves the error) but also *doubles the number of controlled-$U$ applications* on the target. If controlled-$U^{2^{t-1}}$ needs $2^{t-1}$ gates, then adding one more precision qubit doubles the deepest part of the circuit. On today's noisy hardware, this is often the binding constraint.

For our test, we'll pick a simple unitary $U = P(\theta)$ (a phase gate with a chosen $\theta$) acting on a single target qubit prepared in $|1\rangle$. Then:

$$P(\theta)|1\rangle = e^{i\theta}|1\rangle$$

So $2\pi\phi = \theta$, or $\phi = \theta / (2\pi)$. For our test we'll use $\theta = 2\pi \cdot (5/16) = 5\pi/8$, giving $\phi = 5/16 = 0.3125$. This has an exact 4-bit binary representation ($0.0101_2$), which means at $t=4$ precision qubits we should get zero error on an ideal simulator.

*If your class has not covered QPE yet, this test setup will make more sense after your lecture. For now, take on faith that the number we're trying to estimate is $\phi = 5/16$.*


## Setup

In [ ]:
# Standard scientific Python
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

# qBraid
from qbraid.runtime import QbraidProvider

plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

# The exact phase we're trying to estimate
TRUE_PHI = 5 / 16   # = 0.3125
TRUE_THETA = 2 * np.pi * TRUE_PHI  # = 5*pi/8

print(f"True phase: phi = {TRUE_PHI} (exact 4-bit binary: 0.0101)")
print(f"True angle: theta = {TRUE_THETA:.6f} rad")

## Building QPE

We'll construct QPE for our test operator with a configurable number of precision qubits $t$. The target register is a single qubit prepared in $|1\rangle$ (an eigenstate of the phase gate).


In [ ]:
def build_qpe_circuit(t, theta=TRUE_THETA):
    """
    Build a QPE circuit with `t` precision qubits.
    Target register is 1 qubit prepared in |1> (an eigenstate of P(theta)).
    Estimates phi where P(theta)|1> = e^(i * 2*pi*phi) |1>.
    """
    precision = QuantumRegister(t, 'p')
    target = QuantumRegister(1, 'psi')
    creg = ClassicalRegister(t, 'c')
    qc = QuantumCircuit(precision, target, creg)

    # Prepare target in |1> (eigenstate)
    qc.x(target[0])

    # Uniform superposition on precision qubits
    qc.h(precision)

    # Controlled-U^(2^j) applications
    # For P(theta), U^(2^j) = P(2^j * theta)
    for j in range(t):
        power = 2 ** j
        qc.cp(power * theta, precision[j], target[0])

    # Inverse QFT on precision register
    qc.append(QFT(t, inverse=True, do_swaps=True).to_gate(), precision)

    # Measure precision register
    qc.measure(precision, creg)
    return qc


# Sanity-check the circuit at t=4
qc_test = build_qpe_circuit(t=4)
print(f"t=4 circuit depth: {qc_test.depth()}")
print(f"t=4 gate count: {sum(qc_test.count_ops().values())}")
qc_test.draw('mpl', fold=100)

## Extracting phase estimates from measurements

QPE returns measurement bitstrings in a specific ordering: the qubit measured *last* in the QFT-inverted register corresponds to the most significant bit of the phase estimate. We need a helper that converts a bitstring to a phase estimate.


In [ ]:
def bitstring_to_phase(bitstring, t):
    """
    Convert a QPE measurement bitstring to a phase estimate.
    Qiskit returns bitstrings in little-endian order (leftmost char is highest-indexed qubit).
    """
    # Reverse to little-endian, then interpret as fractional binary
    integer = int(bitstring, 2)
    return integer / (2 ** t)


def counts_to_phase_estimate(counts, t):
    """
    Given a dict of {bitstring: count}, return the maximum-likelihood phase estimate.
    """
    total = sum(counts.values())
    weighted = 0.0
    # Use the most probable bitstring as the estimate
    best_bitstring = max(counts, key=counts.get)
    return bitstring_to_phase(best_bitstring, t)


def counts_to_expected_phase(counts, t):
    """
    Return the expected value of phase across the measurement distribution,
    accounting for the circular nature of phase.
    """
    total = sum(counts.values())
    # Compute average of complex exponentials, then extract angle
    z_sum = 0j
    for bitstring, count in counts.items():
        phase = bitstring_to_phase(bitstring, t)
        z_sum += (count / total) * np.exp(2j * np.pi * phase)
    return (np.angle(z_sum) / (2 * np.pi)) % 1.0


# Verify on ideal simulator
sim = AerSimulator()
result = sim.run(transpile(qc_test, sim), shots=4096).result()
counts = result.get_counts()

est_ml = counts_to_phase_estimate(counts, t=4)
est_avg = counts_to_expected_phase(counts, t=4)
print(f"Max-likelihood estimate at t=4: {est_ml} (true: {TRUE_PHI})")
print(f"Circular-mean estimate at t=4: {est_avg:.6f} (true: {TRUE_PHI})")
print(f"Bitstring '0101' probability: {counts.get('0101', 0) / 4096:.3f}")

At $t=4$ the true phase $5/16$ is exactly representable in 4 binary digits, so the ideal simulator should return the bitstring `0101` with probability 1.0 (up to shot noise). If you see anything else, something is off in the circuit construction.

## Precision sweep on the ideal simulator

Now sweep $t$ from 2 to 5 and record the estimation error. On the ideal simulator, error should decrease exponentially.


In [ ]:
def error_vs_true(estimate, true=TRUE_PHI):
    """Return absolute error on the circle (handles wraparound)."""
    d = abs(estimate - true)
    return min(d, 1 - d)  # circular distance


t_range = list(range(2, 6))
SHOTS_SIM = 4096

ideal_errors = []
for t in t_range:
    qc = build_qpe_circuit(t=t)
    result = sim.run(transpile(qc, sim), shots=SHOTS_SIM).result()
    counts = result.get_counts()
    est = counts_to_expected_phase(counts, t)
    err = error_vs_true(est)
    ideal_errors.append(err)
    print(f"t={t}: estimate={est:.6f}, error={err:.6f}")

# Theoretical error bound: 2^-t
theory_errors = [2 ** (-t) for t in t_range]

Perfect. Errors decrease as $2^{-t}$, exactly as theory predicts. At $t=4$ the error is essentially zero because our chosen phase $5/16$ has an exact 4-bit binary representation.

Now let's do the same sweep on real hardware.

## Precision sweep on real hardware

We'll pick one hardware backend (IQM Garnet works well for this because it has a good balance of qubit count and fidelity). Rerun the same $t$ sweep and compare against the ideal curve.

**Note on shots.** With more precision qubits, the measurement distribution spreads out and we need more shots for a stable estimate. But shots cost credits and add to queue time. We use 500 shots per data point as a reasonable trade-off.


In [ ]:
provider = QbraidProvider()

# Instructor: replace with an available backend from your account.
DEVICE_ID = 'iqm_garnet'
device = provider.get_device(DEVICE_ID)

SHOTS_HW = 500

hardware_errors = []
hardware_stats = []
for t in t_range:
    qc = build_qpe_circuit(t=t)

    # Report circuit depth and gate count before submitting
    transpiled = transpile(qc, backend=device.profile.get('qiskit_backend', None), optimization_level=2)
    depth = transpiled.depth()
    n_gates = sum(transpiled.count_ops().values())
    n_2q = sum(v for k, v in transpiled.count_ops().items() if k in ['cx', 'cz', 'iswap', 'ecr'])
    hardware_stats.append({'t': t, 'depth': depth, 'gates': n_gates, '2Q gates': n_2q})

    print(f"t={t}: depth={depth}, gates={n_gates}, 2Q gates={n_2q}. Submitting...")
    job = device.run(qc, shots=SHOTS_HW)
    result = job.result()
    counts = result.data.get_counts()
    est = counts_to_expected_phase(counts, t)
    err = error_vs_true(est)
    hardware_errors.append(err)
    print(f"  -> estimate={est:.4f}, error={err:.4f}")

pd.DataFrame(hardware_stats).set_index('t')

## The payoff plot: ideal vs hardware

Here's the picture that motivates the rest of this notebook.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.semilogy(t_range, theory_errors, 'k--', label='Theoretical bound ($2^{-t}$)', linewidth=2, alpha=0.5)
ax.semilogy(t_range, [max(e, 1e-4) for e in ideal_errors], 'o-', color='#2d7a4f',
            label='Ideal simulator', markersize=12, linewidth=2.5, markeredgecolor='white', markeredgewidth=1.5)
ax.semilogy(t_range, hardware_errors, 's-', color='#c63792',
            label=f'Real hardware ({DEVICE_ID})', markersize=12, linewidth=2.5, markeredgecolor='white', markeredgewidth=1.5)

ax.set_xlabel('Precision qubits (t)', fontsize=12)
ax.set_ylabel('Phase estimation error', fontsize=12)
ax.set_title('QPE precision sweep: ideal vs. real hardware', fontsize=13, pad=15)
ax.set_xticks(t_range)
ax.grid(alpha=0.3, which='both')
ax.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()

Notice the shape. The ideal simulator gives you exponentially decreasing error. The hardware curve typically decreases at first, then plateaus or reverses past a certain $t$. There's an optimal precision qubit count for your specific hardware, and going beyond it makes your answer *worse*, not better.

This is the point no ideal-simulator experiment can teach. QPE precision on NISQ hardware is bounded by circuit-depth noise, not shot noise. The exponential precision improvement that QPE promises requires an exponentially deeper circuit, and exponential-depth circuits die on NISQ hardware.

## Zero-noise extrapolation

Can we do anything about this? Yes, with error mitigation. The simplest and most widely used mitigation technique for measurement-based quantities is Zero-Noise Extrapolation (ZNE).

The idea: run the same circuit at several *noise scale factors* $\lambda \in \{1, 3, 5, ...\}$, where scale factor 1 is the natural noise level and higher factors deliberately inject more noise. Then extrapolate the measured observable back to $\lambda = 0$ (the zero-noise limit) using a polynomial fit.

The way to inject more noise without changing the algorithm is **gate folding**: replace each two-qubit gate $U$ with $U \cdot U^\dagger \cdot U$, which is mathematically the identity times $U$ but physically has three times the noise. Extending to scale factor 5 replaces $U$ with $U \cdot U^\dagger \cdot U \cdot U^\dagger \cdot U$, and so on.

We'll fix $t=4$ (near the optimal-for-noise but where we still care about precision) and apply ZNE with scale factors $\{1, 3, 5\}$.


In [ ]:
from qiskit import transpile

def fold_gates_global(qc, scale_factor):
    """
    Global folding: replace circuit U with (U^-1 U)^n U for scale factor 2n+1.
    Requires scale_factor to be odd (1, 3, 5, ...).
    """
    if scale_factor == 1:
        return qc
    if scale_factor % 2 == 0:
        raise ValueError('Scale factor must be odd for global folding')

    n = (scale_factor - 1) // 2

    # Build the inverse of the measured circuit
    qc_measure = None
    qc_pure = qc.copy()
    # Remove measurements for folding
    qc_pure.remove_final_measurements(inplace=True)

    qc_inv = qc_pure.inverse()

    folded = qc_pure.copy()
    for _ in range(n):
        folded.compose(qc_inv, inplace=True)
        folded.compose(qc_pure, inplace=True)

    # Re-add measurements
    folded.measure_all()
    return folded


# Test that folding preserves the ideal result
qc_base = build_qpe_circuit(t=4)
qc_folded_3 = fold_gates_global(qc_base, scale_factor=3)
qc_folded_5 = fold_gates_global(qc_base, scale_factor=5)

print(f"Scale factor 1: depth={qc_base.depth()}, gates={sum(qc_base.count_ops().values())}")
print(f"Scale factor 3: depth={qc_folded_3.depth()}, gates={sum(qc_folded_3.count_ops().values())}")
print(f"Scale factor 5: depth={qc_folded_5.depth()}, gates={sum(qc_folded_5.count_ops().values())}")

# Verify on simulator that the folded circuits give the same answer as unfolded
for lam, qc in [(1, qc_base), (3, qc_folded_3), (5, qc_folded_5)]:
    result = sim.run(transpile(qc, sim), shots=SHOTS_SIM).result()
    counts_sim = result.get_counts()
    # For folded circuits qiskit may use measure_all which adds ancilla measurements
    # Filter to the first 4 bits
    counts_precision = {}
    for bs, ct in counts_sim.items():
        key = bs[-4:] if len(bs) > 4 else bs
        counts_precision[key] = counts_precision.get(key, 0) + ct
    est = counts_to_expected_phase(counts_precision, t=4)
    print(f"Scale {lam} on ideal simulator: estimate={est:.6f} (should be {TRUE_PHI})")

Folding preserves the ideal answer, as it should. Now run each folded circuit on real hardware and extrapolate.

In [ ]:
SHOTS_ZNE = 500
scale_factors = [1, 3, 5]

zne_data = []
for lam in scale_factors:
    qc = fold_gates_global(build_qpe_circuit(t=4), scale_factor=lam)
    print(f"Running scale factor {lam} on hardware (this takes a few minutes)...")
    job = device.run(qc, shots=SHOTS_ZNE)
    result = job.result()
    counts = result.data.get_counts()

    # Filter to first 4 bits if necessary
    counts_p = {}
    for bs, ct in counts.items():
        key = bs[-4:] if len(bs) > 4 else bs
        counts_p[key] = counts_p.get(key, 0) + ct

    est = counts_to_expected_phase(counts_p, t=4)
    err = error_vs_true(est)
    zne_data.append({'scale_factor': lam, 'estimate': est, 'error': err})
    print(f"  Scale factor {lam}: estimate={est:.4f}, error={err:.4f}")

zne_df = pd.DataFrame(zne_data)
zne_df

## Extrapolate to zero noise

Fit a straight line through the $(\lambda, \text{estimate})$ points and extrapolate to $\lambda = 0$. That extrapolated value is our mitigated estimate.


In [ ]:
# Linear extrapolation
lambdas = np.array([d['scale_factor'] for d in zne_data])
estimates = np.array([d['estimate'] for d in zne_data])

# Fit y = a + b*lambda, extrapolate to lambda=0 means y_mitigated = a
p_linear = np.polyfit(lambdas, estimates, deg=1)
mitigated_linear = p_linear[1]  # intercept

# Richardson extrapolation (quadratic fit through same points)
p_richardson = np.polyfit(lambdas, estimates, deg=2)
mitigated_richardson = np.polyval(p_richardson, 0)

print(f"Raw estimate (scale=1):     {estimates[0]:.4f}  (error: {error_vs_true(estimates[0]):.4f})")
print(f"Linear ZNE:                 {mitigated_linear:.4f}  (error: {error_vs_true(mitigated_linear):.4f})")
print(f"Richardson ZNE:             {mitigated_richardson:.4f}  (error: {error_vs_true(mitigated_richardson):.4f})")
print(f"True value:                 {TRUE_PHI:.4f}")

In [ ]:
# Visualize the extrapolation
fig, ax = plt.subplots(figsize=(9, 6))

# Data points
ax.plot(lambdas, estimates, 'o', color='#c63792', markersize=14, label='Hardware measurements',
        markeredgecolor='white', markeredgewidth=1.5, zorder=3)

# Extrapolation lines
lam_range = np.linspace(-0.5, 5.5, 100)
ax.plot(lam_range, np.polyval(p_linear, lam_range), '--', color='#2d7a4f',
        label=f'Linear fit -> {mitigated_linear:.4f}', linewidth=2)
ax.plot(lam_range, np.polyval(p_richardson, lam_range), ':', color='#1a5285',
        label=f'Richardson fit -> {mitigated_richardson:.4f}', linewidth=2)

# True value
ax.axhline(TRUE_PHI, color='k', linewidth=1.5, alpha=0.4, label=f'True phi = {TRUE_PHI}')
ax.axvline(0, color='k', linewidth=0.5, alpha=0.3)

ax.set_xlabel(r'Noise scale factor $\lambda$', fontsize=12)
ax.set_ylabel('Phase estimate', fontsize=12)
ax.set_title('Zero-noise extrapolation for QPE at t=4', fontsize=13, pad=15)
ax.set_xlim(-0.5, 5.5)
ax.grid(alpha=0.3)
ax.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

ZNE typically recovers a significant fraction of the ideal accuracy, at the cost of a factor of $\lambda_{max}$ more shots (in our case, 3× as many total shots for the mitigated estimate compared to the raw estimate).

**When does ZNE help?** ZNE assumes that the observable's dependence on noise is smooth and low-order-polynomial. That assumption holds for most single-observable measurements but can fail for observables that depend on many correlated qubits or for very noisy circuits where all measurements collapse to random. In practice, ZNE gives you a factor of 2 to 10 improvement in effective circuit depth for the cost of running the circuit at 2 to 3 different scale factors.

## When to use QPE, when to use something else

The precision sweep reveals a hard truth: on today's hardware, QPE is limited to a small number of precision qubits. For our test case, we saw the useful precision plateau at some $t^* < 5$. This has real consequences for what algorithms you can actually run.

QPE is a good fit when:

- You need a *specific known eigenvalue* to good precision, not a general spectrum.
- Your controlled-$U$ can be implemented efficiently in native gates (i.e., $U$ is "close to hardware-native").
- You have access to error mitigation or a device with high enough fidelity to support the required depth.
- The problem has enough symmetry or structure that mitigation techniques work well.

Consider a variational alternative (VQE and friends) when:

- You need the *ground state or lowest few excited states*, not arbitrary eigenvalues.
- Your Hamiltonian is expressible as a sum of Pauli terms (Ising models, molecular Hamiltonians).
- Circuit depth matters more than convergence guarantees.
- You can tolerate iterative classical optimization overhead.

The rule of thumb: QPE gives you a specific number to specific precision, VQE gives you an approximation of a specific eigenstate. On today's hardware, VQE almost always wins for chemistry and materials problems because the depth requirements are lower. QPE becomes competitive as hardware improves and error correction matures.

For the algorithms course, the practical takeaway is: teach both, and be honest about which one you'd actually use in practice today.

## Where to go next

- **Redo the sweep with a phase that is *not* an exact binary fraction.** Set `TRUE_PHI = 1/3` and see how the precision sweep behaves. The convergence pattern is subtler and more instructive than the exact case.
- **Compare mitigation techniques.** ZNE is one option. Randomized compiling and probabilistic error cancellation are others. The `Systems, Hardware & Engineering` notebook in this series covers those in more depth.
- **Apply QPE to a chemistry problem.** The `Quantum Chemistry & Physics` notebook series applies QPE and VQE to $H_2$ ground state estimation and shows the trade-offs on real hardware.


---

**Feedback for the QUEST pedagogy study**

Your feedback informs the IRB-approved QUEST research project on quantum computing pedagogy. Please spend 2 minutes on the following:

1. What was the most useful part of this notebook for your learning?
2. What was the most confusing or under-explained?
3. Which quantum algorithm would you most want to see analyzed this way next?

Please submit your responses via the QUEST portal or reply to your instructor.
